# Volume Profile: Data-Driven Bandwidth Selection for Trading Levels

**Use case:** Identify support/resistance levels from intraday volume profile using KDE.

**Key insight:** Bandwidth selection directly determines how many price levels are detected.
- Silverman (normal reference) → oversmooths → misses real levels in multimodal volume data
- GSJ/Sheather-Jones (data-driven) → adapts to actual price clustering → finds more actionable levels

**Data:** 5 days of 1-minute bars from yfinance for NVDA, TSLA, AAPL, SPY, AMD.


## Method

1. Download 1-min OHLCV bars (last 5 trading days)
2. Compute typical price = (H+L+C)/3 per bar
3. Resample 5000 prices weighted by volume → volume profile
4. Fit Gaussian KDE with bandwidth from each method
5. Find peaks (modes) of the KDE → these are support/resistance levels
6. Overlay on price chart as horizontal lines

The bandwidth determines the resolution: smaller h = more peaks = more levels detected.


In [ ]:
import sys
sys.path.insert(0, 'gsj/src')
import numpy as np
import yfinance as yf
from scipy.signal import find_peaks
from gsj import bandwidth
import matplotlib.pyplot as plt

def get_data(ticker):
    df = yf.download(ticker, period='5d', interval='1m', progress=False)
    if hasattr(df.columns, 'levels'):
        df.columns = df.columns.get_level_values(0)
    df['TypicalPrice'] = (df['High'] + df['Low'] + df['Close']) / 3
    return df

def build_volume_profile(prices, volumes, n_samples=5000):
    probs = volumes / volumes.sum()
    rng = np.random.default_rng(42)
    return prices[rng.choice(len(prices), size=n_samples, p=probs)]

def detect_levels(data, h, min_prominence=0.05):
    x_grid = np.linspace(data.min()*0.999, data.max()*1.001, 1000)
    kde_y = np.zeros_like(x_grid)
    for i in range(0, len(data), 200):
        batch = data[i:i+200]
        diff = x_grid[:, None] - batch[None, :]
        kde_y += np.sum(np.exp(-diff**2 / (2*h**2)), axis=1)
    kde_y /= len(data) * np.sqrt(2*np.pi) * h
    peaks, _ = find_peaks(kde_y, prominence=min_prominence*kde_y.max(), distance=20)
    return x_grid[peaks], x_grid, kde_y

def silverman_bw(data):
    n = len(data)
    A = min(np.std(data, ddof=1), np.subtract(*np.percentile(data, [75,25]))/1.349)
    return 0.9 * A * n**(-1/5)


## NVDA

![Volume Profile NVDA](volume_profile_NVDA.png)

## TSLA

![Volume Profile TSLA](volume_profile_TSLA.png)

## AAPL

![Volume Profile AAPL](volume_profile_AAPL.png)

## SPY

![Volume Profile SPY](volume_profile_SPY.png)

## AMD

![Volume Profile AMD](volume_profile_AMD.png)

## Summary

| Ticker | h (Silverman) | h (GSJ) | Ratio | Levels (Silverman) | Levels (GSJ) | Extra levels |
|--------|---------------|---------|-------|-------------------|--------------|--------------|
| NVDA   | 0.597 | 0.236 | 0.40x | 2 | **4** | +2 |
| TSLA   | 0.678 | 0.395 | 0.58x | 3 | **5** | +2 |
| AAPL   | 0.775 | 0.310 | 0.40x | 3 | **4** | +1 |
| SPY    | 0.704 | 0.324 | 0.46x | 4 | **5** | +1 |
| AMD    | 2.967 | 1.170 | 0.39x | 3 | **4** | +1 |

**Across all 5 stocks, GSJ consistently identifies 1-2 additional support/resistance levels
that Silverman's normal-reference rule merges together.**


## Why This Matters for Trading

1. **Silverman assumes the volume distribution is unimodal** (bell-shaped). Intraday volume profiles are almost NEVER unimodal — price clusters at specific levels where large orders sit.

2. **GSJ adapts to the actual multimodal structure.** When volume clusters at 4 price levels, GSJ finds 4 levels. Silverman might merge 2 nearby levels into one blurry zone.

3. **The extra levels are actionable:**
   - **NVDA** $222.06: a mid-range rotation point between the two extremes that Silverman completely misses
   - **TSLA** $336.88 and $343.60: intermediate levels in a 5-level range structure
   - **AMD** $489.35: a level between two widely separated clusters

4. **The bandwidth ratio (0.39-0.58x) shows the magnitude of Silverman's oversmoothing** on real intraday data. The data-driven bandwidth is 40-60% smaller — this isn't a subtle difference.

5. **Risk management:** More granular levels = tighter stops = better risk/reward. Missing a level means your stop is at the wrong price.

## Interpretation for the Paper

This demo validates the SJ method on a practical 1D application where:
- The data is genuinely multimodal (n=5000, 3-5 modes)
- The consequence of bandwidth choice is directly measurable (number of levels)
- The "ground truth" is visible in the price chart (price respects the detected levels)
- Silverman's normal-reference assumption is clearly violated

No ISE computation needed — the quality metric is "does the detected level correspond to real price action?" Visual inspection confirms the GSJ levels align with actual price consolidation zones.
